# Benchmark de inferência — `winnex-nano`

Braço **winnex-nano** de um par. O outro braço é `transformers`, e roda no notebook irmão
com **o mesmo harness** e a **mesma ordem de células**. A única diferença entre
os dois arquivos é a classe que carregam: `WinnexArm`.

## O que este notebook mede

| bloco | dtype | o que responde | métrica |
|---|---|---|---|
| ambiente | — | a GPU está mesmo sendo usada? | `assert device != "cpu"` |
| tokenizer | — | os dois braços tokenizam igual? | hash dos ids |
| **paridade** | **f32** | os dois calculam a mesma função? | argmax de 50 tokens (gate) |
| perplexidade | f32 | reproduzem o mesmo modelo? | `exp(Σnll/Σtok)` + IC bootstrap |
| MMLU / LAMBADA | f32 | acertam o mesmo? | acurácia + IC Wilson |
| prefill | bf16 | quanto custa processar o prompt? | TTFT, tok/s, **decomposição** |
| decode | bf16 | quanto custa cada token? | ITL em 3 contextos |
| memória | — | quanto RAM/VRAM | anônimo, RSS, VRAM pico |

## O que este notebook NÃO afirma

- **Não é MMLU 5-shot.** É zero-shot: o valor absoluto não é comparável com a
  literatura. A comparação que ele sustenta é winnex vs transformers, no mesmo
  protocolo.
- **Batch = 1.** É o caso de um chat; não é serving com throughput.
- **VRAM vem do `nvidia-smi`, que é do dispositivo**, não do processo. Se outra
  coisa estiver usando a GPU, o número inclui. Está anotado no relatório.
- **Uma placa, um modelo, uma execução.** O IC cobre a variância amostral das
  tarefas, não a variância entre hardware.

## Reprodutibilidade

- Seed fixa e amostra **canônica**: os itens são ordenados por conteúdo antes de
  embaralhar, então duas execuções escolhem os mesmos itens mesmo que o dataset
  tenha sido reordenado. O hash dos itens vai no JSON e os dois braços têm que
  produzir o mesmo.
- 10 iterações, as 2 primeiras descartadas como warmup.
- Reporta média, mediana, P99, IQR e **ruído relativo** `(p99−mediana)/mediana`.
  Uma diferença menor que o ruído não é uma diferença.


## 1. Ambiente

Três verificações antes de instalar qualquer coisa, porque elas decidem se o
benchmark é interpretável: GPU presente, OpenCL carregável (só o braço do
winnex usa), e disco.


In [ ]:
import subprocess, sys, os, shutil, platform
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print("python", platform.python_version(), "|", platform.platform())
print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"))
print(sh("free -g | head -2"))
print(sh("df -h /kaggle/working | tail -1"))


### 1.1 OpenCL

O motor do winnex usa **OpenCL, não CUDA**. Sem um ICD carregável ele lança na
inicialização — que é o comportamento correto, e o oposto de cair para CPU e
reportar um número de GPU que não existe.

O Kaggle não garante o loader; a célula instala se faltar e imprime o que achou.


In [ ]:
if shutil.which("apt-get") and os.geteuid() == 0:
    sh("apt-get install -y -q ocl-icd-libopencl1 ocl-icd-opencl-dev 2>&1 | tail -2")
print("libOpenCL:", sh("ls /usr/lib/x86_64-linux-gnu/libOpenCL.so* 2>&1"))
print("vendors  :", sh("ls /etc/OpenCL/vendors/ 2>&1"))


## 2. Instalação — do PyPI

`winnex-nano==0.1.35`, instalado do PyPI. **Nada é compilado aqui.** Se o
wheel não existir para este Python, o `pip` falha e a célula mostra a mensagem —
o que é diferente de "o código tem um bug", e a diferença importa para saber o
que consertar.


In [ ]:
import subprocess, sys, importlib.metadata as pm
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "winnex-nano==0.1.35", "transformers>=4.40",
                    "datasets", "tokenizers", "huggingface_hub"],
                   capture_output=True, text=True)
print("pip rc =", r.returncode)
if r.returncode != 0:
    print("--- stdout ---"); print(r.stdout[-3000:])
    print("--- stderr ---"); print(r.stderr[-3000:])
    raise SystemExit("instalacao falhou — veja qual wheel faltou para este Python")
for p in ("winnex-nano", "winnex-madhava", "transformers", "torch", "datasets"):
    try: print(f"  {p:<18} {pm.version(p)}")
    except Exception: print(f"  {p:<18} AUSENTE")
import winnex_nano as wn
print("\nwinnex_nano", wn.__version__, "| SIMD AVX2:", wn.compiled_with_simd())


## 3. O harness

`bench_core.py`, o mesmo arquivo nos dois notebooks. Escrito aqui para o
notebook ser autossuficiente no Kaggle.


In [ ]:
CORE = r'''#!/usr/bin/env python3
"""bench_core.py — harness de medição compartilhado pelos dois notebooks.

UM harness para os dois braços. Se o `transformers` e o `winnex-nano` fossem
medidos por códigos diferentes, a diferença entre os números poderia ser a
diferença entre os códigos.

PRINCÍPIOS QUE ESTE ARQUIVO IMPÕE
---------------------------------
1. TODA métrica carrega (valor, n, IC) e o dtype. Número sem os três é saída
   inválida.
2. Os dois braços veem os MESMOS itens. Não "o mesmo n": os mesmos itens,
   identificados por um hash do conteúdo. Amostrar por índice posicional sobre
   uma lista embaralhada não é reprodutível entre execuções em máquinas
   diferentes, e foi o defeito da versão anterior.
3. Falha é resultado. Toda tarefa devolve `status` em {"ok","oom","erro"} e o
   relatório lista as falhas em vez de omiti-las.
4. Gargalo é isolado, não agregado. Prefill é decomposto em custo fixo e custo
   marginal por token; decode é medido em três comprimentos de contexto. É
   isso que mostra ONDE o motor perde, e é simétrico entre os braços.

MEDIÇÕES
--------
  paridade      f32, argmax de 50 tokens gerados. Gate.
  ppl           f32, wikitext-2, exp(Σnll/Σtok), IC bootstrap sobre tokens.
  mmlu/lambada  f32, log-verossimilhança, IC de Wilson. Zero-shot.
  prefill       bf16, TTFT e tokens/s em 512/1024/2048 + decomposição.
  decode        bf16, ITL em ctx 128/512/1024, batch 1.
  memoria       heap anônimo, RSS, VRAM (pico amostrado em thread).
"""
import gc
import hashlib
import math
import os
import statistics
import sys
import time

import numpy as np


# ══ estatística ═════════════════════════════════════════════════════════════

def wilson(k, n, z=1.96):
    """IC de Wilson para proporção. Correto para n pequeno e p perto de 0/1."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    r = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - r) / d, (c + r) / d)


def boot_ratio_ci(num, den, n_boot=4000, alpha=0.05, seed=0, exponentiate=True):
    """IC percentil de Σnum/Σden, reamostrando as unidades independentes.

    A estatística da perplexidade é uma RAZÃO de somas. Reamostrar
    `exp(nll_janela/n_janela)` e tirar a média — o que o harness anterior fazia
    — bootstrapa a média de razões, que é outro estimador, sobre janelas
    sobrepostas que compartilham metade dos tokens.
    """
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64)
    if num.size == 0:
        return (None, None)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, num.size, size=(n_boot, num.size))
    r = num[idx].sum(1) / den[idx].sum(1)
    if exponentiate:
        r = np.exp(r)
    return (float(np.quantile(r, alpha / 2)), float(np.quantile(r, 1 - alpha / 2)))


def pct(v, q):
    v = sorted(v)
    if not v:
        return None
    k = (len(v) - 1) * (q / 100.0)
    f = int(math.floor(k))
    c = min(f + 1, len(v) - 1)
    return v[f] + (v[c] - v[f]) * (k - f)


def describe(times_ms):
    """Média, mediana, P99, IQR, e o ruído relativo (p99-mediana)/mediana.

    O ruído relativo é o que impede reportar como diferença entre motores algo
    que está dentro da dispersão da própria medição.
    """
    if not times_ms:
        return {}
    med = statistics.median(times_ms)
    p99 = pct(times_ms, 99)
    return {"n": len(times_ms),
            "mean_ms": round(statistics.fmean(times_ms), 3),
            "median_ms": round(med, 3),
            "p99_ms": round(p99, 3),
            "iqr_ms": round(pct(times_ms, 75) - pct(times_ms, 25), 3),
            "min_ms": round(min(times_ms), 3),
            "max_ms": round(max(times_ms), 3),
            "ruido_rel": round((p99 - med) / med, 4) if med else None}


def fit_linear(xs, ys):
    """Ajuste y = a + b·x por mínimos quadrados. Devolve (a, b, r2).

    Usado para decompor o prefill: `a` é o custo fixo (que não depende do
    comprimento) e `b` é o custo marginal por token. É a diferença entre "o
    motor é lento" e "o motor paga 300 ms antes de fazer qualquer coisa".
    """
    xs = np.asarray(xs, dtype=np.float64)
    ys = np.asarray(ys, dtype=np.float64)
    if xs.size < 2:
        return (None, None, None)
    b, a = np.polyfit(xs, ys, 1)
    pred = a + b * xs
    ss_res = float(((ys - pred) ** 2).sum())
    ss_tot = float(((ys - ys.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return (float(a), float(b), float(r2))


# ══ reprodutibilidade ══════════════════════════════════════════════════════

def fingerprint(texts):
    """SHA-256 do conteúdo, não da posição.

    Identifica o CONJUNTO de itens que um braço viu. Os dois notebooks têm que
    produzir o mesmo hash; se não produzirem, a comparação de acurácia entre
    eles é entre tarefas diferentes, não entre motores.
    """
    h = hashlib.sha256()
    for t in sorted(texts):
        h.update(t.encode("utf-8", "replace"))
        h.update(b"\x00")
    return h.hexdigest()[:16]


def canonical_sample(rows, n, seed, key):
    """Amostra determinística e independente da ordem do dataset.

    Ordena por `key` antes de embaralhar, para que duas execuções sobre
    revisões diferentes do dataset (mesmo conteúdo, ordem diferente) escolham
    os mesmos itens. A amostra anterior usava índice posicional sobre a ordem
    do dataset — reprodutível na mesma revisão, e só nela.
    """
    import random
    rows = sorted(rows, key=key)
    idx = list(range(len(rows)))
    random.Random(seed).shuffle(idx)
    return [rows[i] for i in idx[:n]]


def tok_fingerprint(arm):
    """Hash dos ids de um texto fixo. Prova que os dois braços tokenizam igual.

    Sem esta verificação, uma diferença de acurácia poderia ser uma diferença
    de tokenização, não de modelo. Já aconteceu neste projeto: o tokenizer do
    HF inseria o token de início e o LAMBADA deu 0/300.
    """
    texto = ("The quick brown fox jumps over the lazy dog. "
             "O rato roeu a roupa do rei de Roma. 0123456789")
    ids = arm.encode(texto, False)
    return {"n_tokens": len(ids), "hash": fingerprint([",".join(map(str, ids))]),
            "ids_head": ids[:8]}


# ══ memória ════════════════════════════════════════════════════════════════

def smaps():
    """(RSS, Anonymous) do processo, em bytes.

    Anônimo é a métrica honesta para o winnex: ele lê o checkpoint por mmap, e
    as páginas de arquivo aparecem no RSS sem serem memória alocada. Contar RSS
    pune o mmap duas vezes.
    """
    rss = anon = 0
    try:
        with open("/proc/self/smaps") as f:
            for line in f:
                if line.startswith("Rss:"):
                    rss += int(line.split()[1]) * 1024
                elif line.startswith("Anonymous:"):
                    anon += int(line.split()[1]) * 1024
    except OSError:
        pass
    return rss, anon


def vram():
    """VRAM usada, em bytes. ATENÇÃO: é do DISPOSITIVO, não do processo.

    Se outra coisa estiver usando a GPU, este número inclui. O notebook diz
    isso no relatório em vez de apresentá-lo como consumo do motor.
    """
    try:
        import subprocess
        o = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10)
        lines = [l for l in o.stdout.strip().splitlines() if l.strip()]
        return sum(int(l) for l in lines) * 2**20
    except Exception:
        return 0


class MemTracker:
    """Amostra o pico de VRAM em thread, porque `nvidia-smi` é caro."""

    def __init__(self):
        self.peak = 0
        self.base = 0
        self._run = False
        self._t = None

    def __enter__(self):
        import threading
        self.base = vram()
        self.peak = self.base
        self._run = True

        def loop():
            while self._run:
                v = vram()
                if v > self.peak:
                    self.peak = v
                time.sleep(0.05)

        self._t = threading.Thread(target=loop, daemon=True)
        self._t.start()
        return self

    def __exit__(self, *a):
        self._run = False
        if self._t:
            self._t.join(timeout=2)


# ══ os dois braços ═════════════════════════════════════════════════════════

class WinnexArm:
    name = "winnex-nano"

    def __init__(self, model_dir, device="gpu", dtype="f32"):
        import winnex_nano as wn
        self.wn = wn
        self.model_dir = model_dir
        self.dtype = dtype
        if dtype == "bf16":
            os.environ["WINNEX_GPU_DTYPE"] = "bf16"
        else:
            os.environ.pop("WINNEX_GPU_DTYPE", None)
        self.cfg = wn.load_config(os.path.join(model_dir, "config.json"))
        self.st = wn.Safetensors.open_dir(model_dir)
        self.eng = wn.ForwardEngine(self.cfg, self.st)
        self.device = "cpu"
        if device == "gpu":
            # enable_gpu() LANÇA quando não há dispositivo. Deixar propagar é o
            # comportamento correto: um benchmark que se diz de GPU não pode
            # continuar em CPU em silêncio.
            self.eng.enable_gpu(os.environ.get("WINNEX_OPENCL_LIB", ""))
            self.device = self.eng.weight_mode
        self.V = self.cfg.vocab_size
        self.max_ctx = int(self.eng.max_context)
        self._tk = None

    def load_tokenizer(self):
        from tokenizers import Tokenizer
        self._tk = Tokenizer.from_file(os.path.join(self.model_dir, "tokenizer.json"))
        return self

    def encode(self, text, add_special_tokens=False):
        return self._tk.encode(text, add_special_tokens=add_special_tokens).ids

    def logits_all(self, ids):
        H = np.ascontiguousarray(self.eng.embed_ids(list(ids)), dtype=np.float32)
        out = self.eng.forward(H, len(ids), True)
        return np.asarray(out, dtype=np.float64).reshape(len(ids), self.V)

    def reset(self):
        self.eng.reset_cache()

    def prefill(self, ids):
        return np.asarray(self.eng.prefill(list(ids)), dtype=np.float64)

    def decode_step(self, token_id):
        return np.asarray(self.eng.step(int(token_id)), dtype=np.float64)

    def sync(self):
        # `step`/`prefill` terminam com leitura bloqueante + clFinish, então o
        # relógio do host já inclui o trabalho de device. Não há o que esperar.
        pass

    def device_time_ms(self):
        """Tempo de KERNEL relatado pelo OpenCL desde o último enable, se houver.

        Serve para separar "o motor é lento" de "o dispositivo é lento e o host
        espera". Devolve None quando o braço não expõe a instrumentação.
        """
        try:
            return float(self.eng.profiling_attn_ms)
        except Exception:
            return None


class HFArm:
    name = "transformers"

    def __init__(self, model_dir, device="gpu", dtype="f32"):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.model_dir = model_dir
        self.dtype = dtype
        self.dev = "cuda" if (device == "gpu" and torch.cuda.is_available()) else "cpu"
        if device == "gpu" and self.dev != "cuda":
            raise RuntimeError("device='gpu' pedido, mas torch.cuda não está disponível")
        td = torch.float32 if dtype == "f32" else torch.bfloat16
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.m = AutoModelForCausalLM.from_pretrained(model_dir, dtype=td).to(self.dev).eval()
        self.device = self.dev
        self.V = self.m.config.vocab_size
        self.max_ctx = int(getattr(self.m.config, "max_position_embeddings", 4096))
        self._past = None

    def load_tokenizer(self):
        return self

    def encode(self, text, add_special_tokens=False):
        return self.tok(text, add_special_tokens=add_special_tokens).input_ids

    def logits_all(self, ids):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([list(ids)], device=self.dev))
        return o.logits[0].float().cpu().numpy().astype(np.float64)

    def reset(self):
        self._past = None

    def prefill(self, ids):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([list(ids)], device=self.dev), use_cache=True)
        self._past = o.past_key_values
        return o.logits[0, -1].float().cpu().numpy().astype(np.float64)

    def decode_step(self, token_id):
        t = self.torch
        with t.no_grad():
            o = self.m(t.tensor([[int(token_id)]], device=self.dev),
                       past_key_values=self._past, use_cache=True)
        self._past = o.past_key_values
        return o.logits[0, -1].float().cpu().numpy().astype(np.float64)

    def sync(self):
        if self.dev == "cuda":
            self.torch.cuda.synchronize()

    def device_time_ms(self):
        return None


# ══ tarefas ════════════════════════════════════════════════════════════════

def _logrow(lg):
    m = lg.max(1, keepdims=True)
    return (lg - m) - np.log(np.exp(lg - m).sum(1, keepdims=True))


def task_parity(arm, prompt_ids, n_gen=50):
    """Gate: argmax de cada token gerado. Devolve tokens e o hash do prompt."""
    arm.reset()
    lg = arm.prefill(prompt_ids)
    got, cur = [], int(np.argmax(lg))
    for _ in range(n_gen):
        got.append(cur)
        lg = arm.decode_step(cur)
        cur = int(np.argmax(lg))
    return {"tokens": got, "prompt_hash": fingerprint([",".join(map(str, prompt_ids))]),
            "prompt_len": len(prompt_ids), "dtype": arm.dtype}


def compare_parity(a, b):
    """Resultado do gate em três estados — nunca um silêncio.

    Devolve status em {"passou","falhou","nao_verificado"} e, quando falha, o
    primeiro índice onde divergiu. Um gate que "não se aplica" é indistinguível
    de um gate que passou, se não for dito.
    """
    if a is None or b is None:
        return {"status": "nao_verificado",
                "motivo": "referencia do outro braco ausente"}
    if a["prompt_hash"] != b["prompt_hash"]:
        return {"status": "nao_verificado",
                "motivo": "prompts diferentes (hash nao bate) — nao comparavel"}
    ta, tb = a["tokens"], b["tokens"]
    n = min(len(ta), len(tb))
    igual = sum(x == y for x, y in zip(ta, tb))
    div = next((i for i in range(n) if ta[i] != tb[i]), None)
    return {"status": "passou" if igual == n and len(ta) == len(tb) else "falhou",
            "n": n, "identicos": igual, "fracao": igual / n if n else 0.0,
            "primeira_divergencia": div}


def task_ppl(arm, seq=512, stride=256, n_windows=40):
    """exp(Σnll/Σtok) em janelas deslizantes, com IC bootstrap sobre tokens."""
    from datasets import load_dataset
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(r["text"].strip() for r in ds if r["text"].strip())
    ids = arm.encode(text, False)
    wins, i = [], 0
    while len(wins) < n_windows and i + seq <= len(ids):
        wins.append(ids[i:i + seq])
        i += stride
    nlls, counts = [], []
    t0 = time.perf_counter()
    for w in wins:
        ls = _logrow(arm.logits_all(w)[:-1])
        v = -ls[np.arange(len(w) - 1), np.asarray(w[1:])]
        nlls.append(v)
        counts.append(np.ones_like(v))
    dt = time.perf_counter() - t0
    nll = np.concatenate(nlls)
    cnt = np.concatenate(counts)
    lo, hi = boot_ratio_ci(nll, cnt)
    return {"task": "ppl/wikitext-2", "status": "ok", "dtype": arm.dtype,
            "ppl": float(math.exp(nll.sum() / cnt.sum())),
            "ppl_ci95": [lo, hi], "n_tokens": int(cnt.sum()),
            "n_janelas": len(wins), "seq": seq, "stride": stride,
            "segundos": round(dt, 1),
            "nota": "IC bootstrap sobre tokens; janelas tem passo 256 e se sobrepoem em 50%"}


def _mcq(arm, items, name, normalize):
    hits, n, erros = 0, 0, 0
    for it in items:
        q = it["question"].strip()
        sc = []
        for ch in it["choices"]:
            prompt = arm.encode(q + "\nAnswer:", False)
            cont = arm.encode(" " + ch.strip(), False)
            ids = prompt + cont
            if len(ids) < 2 or len(ids) > arm.max_ctx - 2:
                sc.append(-1e30)
                continue
            ls = _logrow(arm.logits_all(ids)[:-1])
            s = sum(float(ls[len(prompt) - 1 + j, cont[j]]) for j in range(len(cont)))
            sc.append(s / max(1, len(cont)) if normalize else s)
        if not sc or len(sc) != len(it["choices"]):
            erros += 1
            continue
        hits += int(int(np.argmax(sc)) == it["answer"])
        n += 1
    lo, hi = wilson(hits, n)
    return {"task": f"{name}/{'ll-norm' if normalize else 'll-raw'}",
            "status": "ok" if n else "erro", "dtype": arm.dtype,
            "acc": hits / n if n else None, "acc_ci95": [lo, hi],
            "acertos": hits, "n": n, "itens_com_erro": erros,
            "nota": ("log-verossimilhanca "
                     + ("media por token (variante)" if normalize
                        else "somada, sem normalizar (protocolo MMLU classico)")
                     + "; zero-shot; IC de Wilson")}


def task_mmlu(arm, n=200, seed=0, normalize=True, rows=None):
    """MMLU zero-shot por log-verossimilhança, com amostra estável por conteúdo."""
    if rows is None:
        from datasets import load_dataset
        ds = load_dataset("cais/mmlu", "all", split="test")
        rows = [{"question": ds[i]["question"], "choices": list(ds[i]["choices"]),
                 "answer": int(ds[i]["answer"])} for i in range(len(ds))]
    items = canonical_sample(rows, n, seed, key=lambda r: r["question"])
    r = _mcq(arm, items, "mmlu", normalize)
    r["itens_hash"] = fingerprint([i["question"] for i in items])
    r["fonte"] = f"cais/mmlu all/test, amostra n={len(items)} seed={seed}"
    return r


def task_lambada(arm, n=200, seed=0, rows=None):
    if rows is None:
        from datasets import load_dataset
        ds = load_dataset("lambada", "plain_text", split="test")
        rows = [{"text": ds[i]["text"]} for i in range(len(ds))]
    items = canonical_sample(rows, n, seed, key=lambda r: r["text"])
    hits, n_, skip = 0, 0, 0
    for it in items:
        t = it["text"].strip()
        if " " not in t:
            skip += 1
            continue
        prefix, target = t.rsplit(" ", 1)
        ip, itg = arm.encode(prefix, False), arm.encode(" " + target, False)
        if not itg:
            skip += 1
            continue
        ids = ip + itg
        if len(ids) > arm.max_ctx - 2:
            ids = ids[-(arm.max_ctx - 2):]
            if len(ids) <= len(itg):
                skip += 1
                continue
        lg = arm.logits_all(ids)
        hits += int(int(lg[len(ip) - 1].argmax()) == itg[0])
        n_ += 1
    lo, hi = wilson(hits, n_)
    return {"task": "lambada", "status": "ok" if n_ else "erro", "dtype": arm.dtype,
            "acc": hits / n_ if n_ else None, "acc_ci95": [lo, hi],
            "acertos": hits, "n": n_, "pulados": skip,
            "itens_hash": fingerprint([i["text"] for i in items]),
            "fonte": f"lambada plain_text, amostra n={len(items)} seed={seed}",
            "nota": "argmax exato do primeiro token do alvo; IC de Wilson"}


def _is_oom(msg):
    m = msg.lower()
    return any(k in m for k in ("out of memory", "oom", "cl_mem",
                                "allocate", "memoryerror", "cuda error"))


def task_prefill(arm, lengths=(512, 1024, 2048), iters=10, warmup=2):
    """TTFT e throughput por comprimento, mais a DECOMPOSIÇÃO do custo.

    A decomposição é o que isola o gargalo: ajusta TTFT(n) = fixo + marginal·n.
    Um intercepto alto significa custo que não depende do comprimento (setup,
    transferência inicial); uma inclinação alta significa custo por token. Os
    dois pedem correções diferentes, e a média de tok/s esconde qual é qual.
    """
    out, xs, ys = [], [], []
    for L in lengths:
        ids = list(range(1000, 1000 + min(L, arm.max_ctx - 8)))
        try:
            for _ in range(warmup):
                arm.reset()
                arm.prefill(ids)
                arm.sync()
            times = []
            for _ in range(iters):
                arm.reset()
                t0 = time.perf_counter()
                arm.prefill(ids)
                arm.sync()
                times.append((time.perf_counter() - t0) * 1e3)
            d = describe(times)
            d.update(task="prefill", status="ok", dtype=arm.dtype,
                     n_tokens=len(ids), iters=iters, warmup=warmup,
                     ttft_ms=d["median_ms"],
                     tok_per_s=round(len(ids) / (d["median_ms"] / 1e3), 1))
            out.append(d)
            xs.append(len(ids))
            ys.append(d["median_ms"])
        except (RuntimeError, MemoryError, Exception) as e:
            msg = str(e)[:200]
            out.append({"task": "prefill", "status": "oom" if _is_oom(msg) else "erro",
                        "dtype": arm.dtype, "n_tokens": L, "erro": msg})
            break
    if len(xs) >= 2:
        a, b, r2 = fit_linear(xs, ys)
        decomp = {"custo_fixo_ms": round(a, 2), "custo_por_token_ms": round(b, 4),
                  "r2": round(r2, 5) if r2 == r2 else None,
                  "nota": "ajuste TTFT(n) = fixo + marginal*n; r2 baixo = o modelo linear nao descreve"}
        out.append({"task": "prefill/decomposicao", "status": "ok",
                    "dtype": arm.dtype, **decomp})
    return out


def task_decode(arm, prompt_len=50, n_gen=128, iters=10, warmup=2,
                ctx_list=(128, 512, 1024)):
    """ITL em três comprimentos de contexto, batch 1.

    Medir em três contextos isola o gargalo: se o ITL cresce com o contexto, o
    custo está na atenção/KV; se é plano, está no custo fixo por token (as
    projeções). Um único número de ITL não distingue os dois.
    """
    out = []
    for ctx in ctx_list:
        ctx = min(ctx, arm.max_ctx - n_gen - 8)
        if ctx <= prompt_len:
            continue
        ids = list(range(1000, 1000 + prompt_len))
        try:
            for _ in range(warmup):
                arm.reset()
                lg = arm.prefill(ids)
                cur = int(np.argmax(lg))
                for _ in range(16):
                    cur = int(np.argmax(arm.decode_step(cur)))
                arm.sync()
            runs = []
            for _ in range(iters):
                arm.reset()
                # Preenche até o contexto alvo e decodifica os n_gen seguintes,
                # para que o ITL reflita o KV cache do tamanho declarado.
                lg = arm.prefill(list(range(1000, 1000 + ctx)))
                arm.sync()
                cur = int(np.argmax(lg))
                t0 = time.perf_counter()
                for _ in range(n_gen):
                    lg = arm.decode_step(cur)
                    cur = int(np.argmax(lg))
                arm.sync()
                runs.append((time.perf_counter() - t0) * 1e3 / n_gen)
            d = describe(runs)
            d.update(task="decode", status="ok", dtype=arm.dtype, ctx=ctx,
                     n_gen=n_gen, iters=iters, warmup=warmup, batch=1,
                     itl_ms=d["median_ms"],
                     tok_per_s=round(1000.0 / d["median_ms"], 2) if d["median_ms"] else None)
            out.append(d)
        except (RuntimeError, MemoryError, Exception) as e:
            msg = str(e)[:200]
            out.append({"task": "decode", "status": "oom" if _is_oom(msg) else "erro",
                        "dtype": arm.dtype, "ctx": ctx, "erro": msg})
            break
    return out


def resumo(resultados):
    """Tabela texto com (valor, n, IC) para cada resultado que os tenha."""
    linhas = [f"{'tarefa':<20}{'valor':>11}{'n':>8}  {'IC95':<22}{'dtype':>7} {'status'}"]
    for r in resultados:
        st = r.get("status", "?")
        dt = r.get("dtype", "")
        if st != "ok":
            linhas.append(f"{r['task']:<20}{'—':>11}{'—':>8}  {'—':<22}{dt:>7} {st}")
            continue
        if r.get("acc") is not None:
            lo, hi = r["acc_ci95"]
            linhas.append(f"{r['task']:<20}{r['acc']*100:>10.2f}%{r['n']:>8}  "
                          f"[{lo*100:.1f}, {hi*100:.1f}]{'':<6}{dt:>7} {st}")
        elif r.get("ppl") is not None:
            lo, hi = r["ppl_ci95"]
            linhas.append(f"{r['task']:<20}{r['ppl']:>11.4f}{r['n_tokens']:>8}  "
                          f"[{lo:.2f}, {hi:.2f}]{'':<6}{dt:>7} {st}")
        elif r.get("itl_ms") is not None:
            linhas.append(f"{r['task']} ctx={r['ctx']:<4}{r['itl_ms']:>10.2f}"
                          f"{r['n']:>8}  P99={r['p99_ms']:<14.1f}{dt:>7} {st}")
        elif r.get("ttft_ms") is not None:
            linhas.append(f"{r['task']} n={r['n_tokens']:<5}{r['ttft_ms']:>9.1f}"
                          f"{r['n']:>8}  P99={r['p99_ms']:<14.1f}{dt:>7} {st}")
        elif r.get("custo_fixo_ms") is not None:
            linhas.append(f"{r['task']:<20}{r['custo_fixo_ms']:>11.1f}{'':>8}  "
                          f"marginal {r['custo_por_token_ms']:.4f} ms/tok{'':<1}{dt:>7} {st}")
        else:
            linhas.append(f"{r['task']:<20}{'—':>11}{'—':>8}  {'—':<22}{dt:>7} {st}")
    return "\n".join(linhas)
'''
open('bench_core.py','w').write(CORE)
import bench_core
print('harness carregado:', len(CORE), 'bytes')


## 4. Modelo

Qwen2.5-0.5B-Instruct: 1,0 GB em f32, cabe na T4 **em f32** — condição para o
gate de paridade e a PPL compararem motores sem a confusão do arredondamento
bf16.

Trocar por um Llama-3.2 exige `winnex-nano >= 0.1.35`, que passou a honrar
`rope_scaling`; antes disso o motor rodava um modelo diferente sem avisar.


In [ ]:
import os
from huggingface_hub import snapshot_download
MODELO_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MODELO = "/kaggle/working/modelo"
if not os.path.isdir(MODELO):
    snapshot_download(MODELO_ID, local_dir=MODELO,
                      allow_patterns=["*.json", "*.safetensors", "*.txt", "tokenizer*"])
arqs = [f for f in os.listdir(MODELO) if os.path.isfile(os.path.join(MODELO, f))]
print(sorted(arqs))
print("tamanho:", round(sum(os.path.getsize(os.path.join(MODELO,f)) for f in arqs)/2**30, 2), "GB")


## 5. Memória: linha de base (ponto 1 de 3)

Antes de carregar o modelo. O heap anônimo é a métrica honesta para o winnex:
ele lê o checkpoint por mmap, e páginas de arquivo aparecem no RSS sem serem
memória alocada pelo processo.


In [ ]:
import gc, bench_core
gc.collect()
rss0, anon0 = bench_core.smaps()
vram0 = bench_core.vram()
BASE = {"rss_gb": rss0/2**30, "anon_gb": anon0/2**30, "vram_gb": vram0/2**30}
print(f"baseline  RSS={BASE['rss_gb']:.3f} GB  anonimo={BASE['anon_gb']:.3f} GB  "
      f"VRAM={BASE['vram_gb']:.3f} GB")
print("(VRAM e do dispositivo, nao do processo)")


## 6. Carga do modelo (ponto 2 de 3)

`WinnexArm` com `device="gpu"`, `dtype="f32"`.

O `assert` não é decorativo: se o braço cair para CPU, o bloco de desempenho
mediria CPU e o relatório estaria errado sem parecer errado.


In [ ]:
import time, bench_core
t0 = time.perf_counter()
arm = bench_core.WinnexArm(MODELO, device="gpu", dtype="f32").load_tokenizer()
t_load = time.perf_counter() - t0
rss1, anon1 = bench_core.smaps()
CARGA = {"load_s": round(t_load, 2),
         "anon_delta_gb": round((anon1-anon0)/2**30, 3),
         "vram_delta_gb": round((bench_core.vram()-vram0)/2**30, 3)}
print(f"carga: {t_load:.2f} s | modo: {arm.device} | dtype: {arm.dtype}")
print(f"  anonimo +{CARGA['anon_delta_gb']:.3f} GB | VRAM +{CARGA['vram_delta_gb']:.3f} GB")
print(f"  vocab={arm.V}  contexto_max={arm.max_ctx}")
assert arm.device != "cpu", "braco caiu para CPU — desempenho nao seria interpretavel"

# Prova de que os dois bracos tokenizam igual. Sem isto, uma diferenca de
# acuracia poderia ser diferenca de tokenizacao: ja aconteceu neste projeto
# (o tokenizer do HF inserindo o token de inicio, LAMBADA 0/300).
TOKF = bench_core.tok_fingerprint(arm)
print("tokenizer:", TOKF)


## 7. GATE: paridade de logits (f32)

Prompt de 50 tokens, 50 tokens gerados, argmax de cada um.

**Se este gate falhar, nada abaixo vale** — é por isso que ele vem antes da
qualidade: medir a acurácia de um motor que não reproduz o outro mede o motor,
não o modelo.

O gate tem **três estados**, e o terceiro é o que costuma ser esquecido:

- `passou` — 50/50 idênticos
- `falhou` — mostra o índice da primeira divergência
- `nao_verificado` — a referência do braço `transformers` não está aqui, ou o hash
  do prompt não bate

"não verificado" **não** é "passou". O JSON registra qual dos três ocorreu.

Para cruzar: rode o outro notebook e adicione o `referencia_paridade.json` dele
como *Add Data → Your Work → Output*, ou copie o arquivo para
`/kaggle/working/`.


In [ ]:
import time, json, os, bench_core
PROMPT = ("The history of the Roman Empire is a long and complicated story "
          "that begins with the founding of the city")
prompt_ids = arm.encode(PROMPT, False)[:50]
print("prompt:", len(prompt_ids), "tokens")

t0 = time.perf_counter()
PAR = bench_core.task_parity(arm, prompt_ids, n_gen=50)
print(f"50 tokens em {time.perf_counter()-t0:.1f} s ->", PAR["tokens"][:12])
print("hash do prompt:", PAR["prompt_hash"])

CAND = ["/kaggle/working/referencia_paridade.json", "referencia_paridade.json"]
REF = None
for c in CAND:
    if os.path.exists(c):
        d = json.load(open(c))
        if d.get("arm") != arm.name:
            REF = d
            break
GATE = bench_core.compare_parity(PAR, REF)
print("\nGATE:", json.dumps(GATE, indent=2))
if GATE["status"] == "falhou":
    raise AssertionError(f"GATE FALHOU: diverge no token {GATE['primeira_divergencia']} "
                         f"({GATE['identicos']}/{GATE['n']} identicos)")
json.dump(PAR, open("referencia_paridade.json", "w"), indent=1)
print("referencia gravada")


## 8. Perplexidade de corpus (f32)

Wikitext-2 test, janelas de 512 com passo 256. `PPL = exp(Σnll/Σtok)`.

O IC é bootstrap sobre **tokens**, não sobre janelas: as janelas se sobrepõem em
50% e compartilham metade dos tokens, então tratá-las como amostras
independentes produz um intervalo que não se sustenta. A estatística é a razão
de somas, não a média de razões.


In [ ]:
import bench_core
r_ppl = bench_core.task_ppl(arm, seq=512, stride=256, n_windows=40)
lo, hi = r_ppl["ppl_ci95"]
print(f"PPL = {r_ppl['ppl']:.6f}   IC95 = [{lo:.4f}, {hi:.4f}]")
print(f"n = {r_ppl['n_tokens']} tokens | {r_ppl['n_janelas']} janelas "
      f"seq={r_ppl['seq']} stride={r_ppl['stride']} | {r_ppl['segundos']} s")
print("dtype:", r_ppl["dtype"])


## 9. Acurácia zero-shot (f32)

MMLU e LAMBADA por **log-verossimilhança**: pontua cada alternativa pelos
log-probs da continuação e escolhe a maior. É o protocolo do MMLU original.

A geração greedy fica de fora: ela aceita "a primeira letra A–D que aparecer",
é sensível a formatação, e não é comparável com nada publicado.

Duas leituras do MMLU, na mesma passagem:
- `ll-raw` — soma crua dos log-probs. **É o protocolo clássico.**
- `ll-norm` — a mesma soma dividida pelo número de tokens da alternativa.
  Variante que favorece alternativas curtas; **não é o número publicado.**
  Reportar as duas evita escolher uma depois de ver o resultado.

A amostra é **canônica** (seed 0, ordenada por conteúdo antes de
embaralhar), então o hash dos itens é idêntico nos dois notebooks. Se não for,
a comparação é entre tarefas diferentes, e o JSON mostra isso.

n=200 dá IC de cerca de ±6 pontos numa acurácia perto de 35%. Diferenças
menores não são detectáveis.


In [ ]:
import bench_core
r_mmlu_raw  = bench_core.task_mmlu(arm, n=200, seed=0, normalize=False)
r_mmlu_norm = bench_core.task_mmlu(arm, n=200, seed=0, normalize=True)
r_lambada   = bench_core.task_lambada(arm, n=200, seed=0)
for r in (r_mmlu_raw, r_mmlu_norm, r_lambada):
    lo, hi = r["acc_ci95"]
    print(f"  {r['task']:<16} {r['acc']*100:5.1f}%  n={r['n']:<4} "
          f"IC95=[{lo*100:.1f}, {hi*100:.1f}]  status={r['status']}")
    print(f"      itens_hash={r.get('itens_hash')}")
print("\nZero-shot: nao comparar o valor absoluto com MMLU 5-shot da literatura.")


## 10. Prefill — throughput e decomposição (bf16)

Recarrega em **bf16**, o dtype em que o modelo seria servido. O dtype vai em
cada resultado.

Mede TTFT e tokens/s em 512 / 1024 / 2048 tokens, e **decompõe** o custo:

```
TTFT(n) = custo_fixo + custo_por_token · n
```

A decomposição é o que isola o gargalo. Um intercepto alto é custo que não
depende do comprimento (setup, primeira transferência); uma inclinação alta é
custo por token. Os dois pedem correções diferentes, e a média de tok/s esconde
qual é qual. O `r2` do ajuste vai no resultado — se for baixo, o modelo linear
não descreve a curva e a decomposição não vale.


In [ ]:
import gc, bench_core
del arm
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception: pass

arm16 = bench_core.WinnexArm(MODELO, device="gpu", dtype="bf16").load_tokenizer()
print("modo:", arm16.device, "| dtype:", arm16.dtype)

r_prefill = bench_core.task_prefill(arm16, lengths=(512, 1024, 2048), iters=10, warmup=2)
for r in r_prefill:
    st = r.get("status")
    if st != "ok":
        print(f"  {r['task']:<22} {r.get('n_tokens','')} -> {st}: {r.get('erro','')[:90]}")
    elif r["task"] == "prefill/decomposicao":
        print(f"  decomposicao: fixo={r['custo_fixo_ms']:.1f} ms  "
              f"marginal={r['custo_por_token_ms']:.4f} ms/tok  r2={r['r2']}")
    else:
        print(f"  {r['n_tokens']:>5} tok: TTFT {r['ttft_ms']:9.1f} ms  "
              f"{r['tok_per_s']:9.1f} tok/s  P99 {r['p99_ms']:.1f}  ruido {r['ruido_rel']}")


## 11. Decode — latência em três contextos (bf16)

Prompt de 50 tokens, 128 tokens gerados, **batch = 1**. Métrica: **ITL**
(inter-token latency, ms/token) e tokens/s.

Medido em **ctx = 128, 512 e 1024**, e é isso que isola o gargalo: se o ITL
cresce com o contexto, o custo está na atenção/KV cache; se é plano, está no
custo fixo por token (as projeções). Um único número de ITL não distingue os
dois casos, e os dois pedem correções diferentes.

O P99 importa mais que a média: é o que decide se um timeout derruba uma
requisição em 100.


In [ ]:
import bench_core
r_decode = bench_core.task_decode(arm16, prompt_len=50, n_gen=128, iters=10,
                                  warmup=2, ctx_list=(128, 512, 1024))
for r in r_decode:
    if r.get("status") != "ok":
        print(f"  ctx={r.get('ctx')} -> {r['status']}: {r.get('erro','')[:100]}")
        continue
    print(f"  ctx={r['ctx']:>5}: ITL {r['itl_ms']:7.2f} ms/tok  "
          f"({r['tok_per_s']:7.2f} tok/s)  media {r['mean_ms']:.2f}  "
          f"P99 {r['p99_ms']:.2f}  ruido {r['ruido_rel']}")
print("\nSe o ITL cresce com ctx, o gargalo e atencao/KV; se e plano, e o custo")
print("fixo por token das projecoes.")


## 12. Memória: pico durante a geração (ponto 3 de 3)

- **Heap anônimo** — RAM realmente alocada pelo processo. Ignora páginas de
  arquivo do mmap, que o SO pode descartar. É a métrica honesta para o winnex.
- **RSS** — residente total; mostra o impacto do mmap no sistema.
- **VRAM pico** — amostrada em thread durante a geração, incluindo KV cache.
  **Vem do `nvidia-smi`, que é do dispositivo**: inclui qualquer outro processo
  na GPU. O relatório diz isso em vez de apresentá-lo como consumo do motor.


In [ ]:
import bench_core
with bench_core.MemTracker() as mt:
    ids = list(range(1000, 1000 + min(512, arm16.max_ctx - 40)))
    arm16.reset()
    lg = arm16.prefill(ids)
    cur = int(lg.argmax())
    for _ in range(64):
        lg = arm16.decode_step(cur)
        cur = int(lg.argmax())
    arm16.sync()
rss2, anon2 = bench_core.smaps()
PICO = {"anon_gb": anon2/2**30, "rss_gb": rss2/2**30,
        "vram_peak_gb": max(0.0, (mt.peak - BASE["vram_gb"]*2**30))/2**30}
print(f"heap anonimo  {PICO['anon_gb']:7.3f} GB  (baseline {BASE['anon_gb']:.3f})")
print(f"RSS           {PICO['rss_gb']:7.3f} GB")
print(f"VRAM pico     {PICO['vram_peak_gb']:7.3f} GB  (do dispositivo)")
print("anonimo e a metrica honesta: o mmap do checkpoint entra no RSS sem ser")
print("memoria alocada pelo processo.")


## 13. Limpeza e vazamento

Libera o motor, força coleta e mede de novo. O delta depois da limpeza é o
vazamento. **Deve ser ~0**: memória que sobrevive ao `del` é memória que um
servidor não recupera entre requisições.


In [ ]:
import gc, time, bench_core
del arm16
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
except Exception: pass
time.sleep(1)
rss3, anon3 = bench_core.smaps()
VAZ = {"anon_gb": (anon3-anon0)/2**30, "vram_gb": (bench_core.vram()-vram0)/2**30}
print(f"apos limpeza: anonimo {anon3/2**30:.3f} GB (baseline {anon0/2**30:.3f})")
print(f"vazamento anonimo: {VAZ['anon_gb']:+.3f} GB")
print(f"vazamento VRAM   : {VAZ['vram_gb']:+.3f} GB")


## 14. Exportar

Um JSON por braço, com **todos** os números, o dtype de cada um, o `status`, e
as falhas — que são resultado, não algo a omitir.

No Kaggle ele fica em `/kaggle/working/` e aparece em *Output*.


In [ ]:
import json, time, platform
RES = [r_ppl, r_mmlu_raw, r_mmlu_norm, r_lambada, *r_prefill, *r_decode]
saida = {
    "arm": arm.name,
    "pacote": "winnex-nano 0.1.35" if arm.name == "winnex-nano" else "transformers",
    "modelo": MODELO_ID,
    "maquina": platform.platform(),
    "python": platform.python_version(),
    "data": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "tokenizer": TOKF,
    "gate": GATE,
    "memoria": {"baseline": BASE, "carga": CARGA, "pico": PICO, "vazamento": VAZ},
    "resultados": RES,
}
# Contagem explicita de falhas: o relatorio tem que mostrar o que nao funcionou.
falhas = [r for r in RES if r.get("status") not in (None, "ok")]
saida["resumo_falhas"] = {"n": len(falhas),
    "itens": [{"task": r["task"], "status": r.get("status"),
                "erro": r.get("erro", "")[:200]} for r in falhas]}
ARQ = "resultado_winnex_nano.json"
json.dump(saida, open(ARQ, "w"), indent=2, default=float)
print("gravado:", ARQ)
print("falhas:", len(falhas))


### 14.1 Tabela resumo

Uma linha por métrica, com valor, n, IC, dtype e status. É o que vai para o
relatório — a saída crua não carrega n nem IC.


In [ ]:
import bench_core
print(bench_core.resumo(RES))
print("\nGATE de paridade:", GATE["status"])
if GATE["status"] != "passou":
    print("  ->", GATE.get("motivo") or f"diverge em {GATE.get('primeira_divergencia')}")
